In [ ]:
"""
Waste Classification v3 — EfficientNetV2S
"""

# ───────────────────────── imports ─────────────────────────
import os
import gc
import warnings
import time

warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    BatchNormalization,
    Input,
    Add,
    Concatenate,
    Multiply,
    Reshape,
    Layer,
)
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    LearningRateScheduler,
)
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

try:
    from tqdm import tqdm
except ImportError:
    class tqdm:
        def __init__(self, iterable=None, **kw):
            self._it = iterable
        def __iter__(self):
            return iter(self._it)
        def __enter__(self):
            return self
        def __exit__(self, *_):
            pass
        def update(self, n=1):
            pass

# ─────────────── GPU / performance setup ───────────────────
tf.config.optimizer.set_jit(True)
print("✓ XLA JIT compilation enabled")

try:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("✓ Mixed precision (float16) enabled")
except Exception:
    print("⚠ Mixed precision not available, using float32")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
    print(f"✓ Found {len(gpus)} GPU(s): {[g.name for g in gpus]}")
else:
    print("⚠ No GPU detected — training will be slow on CPU")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ───────────────────── configuration ───────────────────────
DATASET_PATH = "/kaggle/input/datasets/idly1234/wastecnn/Data"

IMG_SIZE = 260          # [MEM-OPT] reduced from 300 — saves ~25% VRAM
BATCH_SIZE = 16         # [MEM-OPT] reduced from 32 — safe for 16 GB GPUs
PHASE1_EPOCHS = 12
PHASE2_EPOCHS = 25
PHASE3_EPOCHS = 15
PHASE1_LR = 1e-3
PHASE2_LR = 5e-6
PHASE3_LR = 1e-6
FINE_TUNE_AT_P2 = -50
FINE_TUNE_AT_P3 = -100
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0
WARMUP_EPOCHS = 3
WEIGHT_DECAY = 1e-4
NUM_PARALLEL = tf.data.AUTOTUNE

# ───────────────────── label maps ──────────────────────────
LEVEL1_LABELS = {"Organic": 0, "NonOrganic": 1}
LEVEL2_LABELS = {"Food": 0, "Plastic": 1, "Metal": 2, "Paper": 3, "Glass": 4}
LEVEL3_LABELS = {
    "PET": 0,
    "HDPE": 1,
    "Aluminum": 2,
    "Steel": 3,
    "Newspaper": 4,
    "Cardboard": 5,
    "ClearGlass": 6,
    "BrownGlass": 7,
}

LEVEL1_NAMES = {v: k for k, v in LEVEL1_LABELS.items()}
LEVEL2_NAMES = {v: k for k, v in LEVEL2_LABELS.items()}
LEVEL3_NAMES = {v: k for k, v in LEVEL3_LABELS.items()}


# ──────────────────── data loading ─────────────────────────
# [MEM-OPT] Only collect file paths + labels — NO images in RAM
def scan_dataset(dataset_path: str):
    """Walk the dataset directory and return file paths + multi-level labels.
    
    Unlike v3, this does NOT load images into RAM. It only stores paths.
    Memory usage: ~1 KB per image (path string) vs ~260×260×3×4 = 810 KB/image.
    """
    paths, lvl1, lvl2, lvl3 = [], [], [], []

    # Valid image magic bytes: JPEG(FFD8), PNG(89504E47), BMP(424D), GIF(474946)
    VALID_MAGIC = {
        b'\xff\xd8': 'jpeg',
        b'\x89PNG': 'png', 
        b'BM': 'bmp',
        b'GIF': 'gif',
    }

    def is_valid_image(filepath):
        """Check file header bytes to verify it's a real image."""
        try:
            with open(filepath, 'rb') as f:
                header = f.read(4)
            if len(header) < 2:
                return False
            for magic in VALID_MAGIC:
                if header[:len(magic)] == magic:
                    return True
            return False
        except Exception:
            return False

    all_files = []
    skipped = 0
    for root, _dirs, files in os.walk(dataset_path):
        for fname in files:
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".gif")):
                fpath = os.path.join(root, fname)
                if is_valid_image(fpath):
                    all_files.append((root, fname))
                else:
                    skipped += 1

    if skipped:
        print(f"⚠ Skipped {skipped} corrupt/invalid files")

    for root, fname in tqdm(all_files, desc="Scanning dataset", unit="img"):
        path = os.path.join(root, fname)
        parts = root.replace("\\", "/").split("/")

        if "Organic" in parts:
            lvl1.append(LEVEL1_LABELS["Organic"])
            lvl2.append(LEVEL2_LABELS["Food"])
            lvl3.append(0)
        elif "NonOrganic" in parts:
            lvl1.append(LEVEL1_LABELS["NonOrganic"])

            if "Plastic" in parts:
                lvl2.append(LEVEL2_LABELS["Plastic"])
                if "PET" in parts:
                    lvl3.append(LEVEL3_LABELS["PET"])
                elif "HDPE" in parts:
                    lvl3.append(LEVEL3_LABELS["HDPE"])
                else:
                    lvl3.append(LEVEL3_LABELS["PET"])

            elif "Metal" in parts:
                lvl2.append(LEVEL2_LABELS["Metal"])
                if "Aluminum" in parts:
                    lvl3.append(LEVEL3_LABELS["Aluminum"])
                elif "Steel" in parts:
                    lvl3.append(LEVEL3_LABELS["Steel"])
                else:
                    lvl3.append(LEVEL3_LABELS["Aluminum"])

            elif "Paper" in parts:
                lvl2.append(LEVEL2_LABELS["Paper"])
                if "Newspaper" in parts:
                    lvl3.append(LEVEL3_LABELS["Newspaper"])
                elif "Cardboard" in parts:
                    lvl3.append(LEVEL3_LABELS["Cardboard"])
                else:
                    lvl3.append(LEVEL3_LABELS["Newspaper"])

            elif "Glass" in parts:
                lvl2.append(LEVEL2_LABELS["Glass"])
                if "ClearGlass" in parts:
                    lvl3.append(LEVEL3_LABELS["ClearGlass"])
                elif "BrownGlass" in parts:
                    lvl3.append(LEVEL3_LABELS["BrownGlass"])
                else:
                    lvl3.append(LEVEL3_LABELS["ClearGlass"])
            else:
                continue
        else:
            continue

        paths.append(path)

    print(f"✓ Found {len(paths)} images in {dataset_path}")

    lvl1_arr, lvl2_arr, lvl3_arr = np.array(lvl1), np.array(lvl2), np.array(lvl3)
    print("\n📊 Class Distribution:")
    print("  Level-1 (Organic/NonOrganic):")
    for name, idx in LEVEL1_LABELS.items():
        count = np.sum(lvl1_arr == idx)
        print(f"    {name}: {count} ({count / len(lvl1_arr):.1%})")
    print("  Level-2 (Material):")
    for name, idx in LEVEL2_LABELS.items():
        count = np.sum(lvl2_arr == idx)
        print(f"    {name}: {count} ({count / len(lvl2_arr):.1%})")
    print("  Level-3 (Sub-type):")
    for name, idx in LEVEL3_LABELS.items():
        count = np.sum(lvl3_arr == idx)
        print(f"    {name}: {count} ({count / len(lvl3_arr):.1%})")

    return np.array(paths), lvl1_arr, lvl2_arr, lvl3_arr


# ────────── Image loading as tf.data operation ─────────────
# [MEM-OPT] Load and decode images on-the-fly from disk

def load_and_preprocess(path, y1, y2, y3):
    """Load a single image from disk inside the tf.data pipeline."""
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)  # 0-255 range for EfficientNetV2
    return img, y1, y2, y3


# ────────── GPU-accelerated augmentation via tf.image ──────
@tf.function
def gpu_augment(image):
    """Apply random augmentations using tf.image ops (GPU-accelerated)."""
    if tf.random.uniform(()) > 0.5:
        image = tf.image.rot90(image, k=tf.random.uniform((), 0, 4, dtype=tf.int32))

    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.3 * 255)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.3)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    image = tf.image.random_hue(image, max_delta=0.05)

    crop_size = tf.random.uniform((), minval=0.75, maxval=1.0)
    crop_h = tf.cast(tf.cast(IMG_SIZE, tf.float32) * crop_size, tf.int32)
    crop_w = crop_h
    image = tf.image.random_crop(image, size=[crop_h, crop_w, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])

    image = tf.clip_by_value(image, 0.0, 255.0)
    return image


@tf.function
def tf_mixup(images, y1, y2, y3, alpha=0.2):
    """GPU-accelerated MixUp."""
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform((), minval=0.0, maxval=1.0)

    perm = tf.random.shuffle(tf.range(batch_size))
    images_b = tf.gather(images, perm)
    y1_b = tf.gather(y1, perm)
    y2_b = tf.gather(y2, perm)
    y3_b = tf.gather(y3, perm)

    mixed_images = lam * images + (1.0 - lam) * images_b
    mixed_y1 = lam * y1 + (1.0 - lam) * y1_b
    mixed_y2 = lam * y2 + (1.0 - lam) * y2_b
    mixed_y3 = lam * y3 + (1.0 - lam) * y3_b
    return mixed_images, mixed_y1, mixed_y2, mixed_y3


@tf.function
def tf_cutmix(images, y1, y2, y3, alpha=1.0):
    """GPU-accelerated CutMix."""
    batch_size = tf.shape(images)[0]
    H = tf.shape(images)[1]
    W = tf.shape(images)[2]

    lam = tf.random.uniform(())

    cut_ratio = tf.sqrt(1.0 - lam)
    cut_h = tf.cast(tf.cast(H, tf.float32) * cut_ratio, tf.int32)
    cut_w = tf.cast(tf.cast(W, tf.float32) * cut_ratio, tf.int32)
    cy = tf.random.uniform((), 0, H, dtype=tf.int32)
    cx = tf.random.uniform((), 0, W, dtype=tf.int32)
    y1_box = tf.clip_by_value(cy - cut_h // 2, 0, H)
    y2_box = tf.clip_by_value(cy + cut_h // 2, 0, H)
    x1_box = tf.clip_by_value(cx - cut_w // 2, 0, W)
    x2_box = tf.clip_by_value(cx + cut_w // 2, 0, W)

    rows = tf.range(H)
    cols = tf.range(W)
    row_mask = tf.logical_and(rows >= y1_box, rows < y2_box)
    col_mask = tf.logical_and(cols >= x1_box, cols < x2_box)
    mask = tf.cast(
        tf.logical_not(tf.logical_and(row_mask[:, None], col_mask[None, :])),
        tf.float32,
    )
    mask = mask[:, :, None]

    perm = tf.random.shuffle(tf.range(batch_size))
    images_b = tf.gather(images, perm)
    y1_b = tf.gather(y1, perm)
    y2_b = tf.gather(y2, perm)
    y3_b = tf.gather(y3, perm)

    mixed_images = images * mask + images_b * (1.0 - mask)

    actual_lam = 1.0 - tf.cast((y2_box - y1_box) * (x2_box - x1_box), tf.float32) / tf.cast(H * W, tf.float32)
    mixed_y1 = actual_lam * y1 + (1.0 - actual_lam) * y1_b
    mixed_y2 = actual_lam * y2 + (1.0 - actual_lam) * y2_b
    mixed_y3 = actual_lam * y3 + (1.0 - actual_lam) * y3_b
    return mixed_images, mixed_y1, mixed_y2, mixed_y3


# ──────────── tf.data pipeline builders ────────────────────
# [MEM-OPT] Reads images from DISK — no NumPy array in RAM

def build_train_dataset(paths, y1, y2, y3, batch_size, use_mixup=True):
    """
    Build a memory-efficient tf.data training pipeline.
    
    Pipeline: paths → read_file → decode → resize → shuffle → batch →
              augment (GPU) → mixup/cutmix (GPU) → prefetch
    """
    # One-hot encode labels
    y1_oh = to_categorical(y1, num_classes=2).astype(np.float32)
    y2_oh = to_categorical(y2, num_classes=5).astype(np.float32)
    y3_oh = to_categorical(y3, num_classes=8).astype(np.float32)

    dataset = tf.data.Dataset.from_tensor_slices((paths, y1_oh, y2_oh, y3_oh))
    dataset = dataset.shuffle(buffer_size=min(len(paths), 5000), seed=SEED,
                              reshuffle_each_iteration=True)

    # [MEM-OPT] Load images from disk on-the-fly (parallel I/O)
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=NUM_PARALLEL)
    dataset = dataset.batch(batch_size, drop_remainder=True)

    # GPU augmentation
    def augment_batch(imgs, l1, l2, l3):
        imgs = tf.map_fn(gpu_augment, imgs, fn_output_signature=tf.float32)
        return imgs, l1, l2, l3

    if use_mixup:
        def augment_and_mix(imgs, l1, l2, l3):
            imgs = tf.map_fn(gpu_augment, imgs, fn_output_signature=tf.float32)
            if tf.random.uniform(()) < 0.5:
                imgs, l1, l2, l3 = tf_mixup(imgs, l1, l2, l3, alpha=MIXUP_ALPHA)
            else:
                imgs, l1, l2, l3 = tf_cutmix(imgs, l1, l2, l3, alpha=CUTMIX_ALPHA)
            return imgs, l1, l2, l3

        dataset = dataset.map(augment_and_mix, num_parallel_calls=NUM_PARALLEL)
    else:
        dataset = dataset.map(augment_batch, num_parallel_calls=NUM_PARALLEL)

    # Format output for multi-output model
    def format_output(imgs, l1, l2, l3):
        return imgs, {
            "organic_output": l1,
            "material_output": l2,
            "subtype_output": l3,
        }

    dataset = dataset.map(format_output, num_parallel_calls=NUM_PARALLEL)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


def build_eval_dataset(paths, y1, y2, y3, batch_size):
    """Build a memory-efficient tf.data evaluation pipeline (no augmentation)."""
    y1_oh = to_categorical(y1, num_classes=2).astype(np.float32)
    y2_oh = to_categorical(y2, num_classes=5).astype(np.float32)
    y3_oh = to_categorical(y3, num_classes=8).astype(np.float32)

    dataset = tf.data.Dataset.from_tensor_slices((paths, y1_oh, y2_oh, y3_oh))

    # [MEM-OPT] Load from disk, no .cache() — avoids RAM duplication
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=NUM_PARALLEL)

    def format_output(imgs, l1, l2, l3):
        return imgs, {
            "organic_output": l1,
            "material_output": l2,
            "subtype_output": l3,
        }

    dataset = dataset.map(format_output, num_parallel_calls=NUM_PARALLEL)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


# ──────────── SE Attention Block ───────────────────────────
class SEBlock(Layer):
    """Squeeze-and-Excitation attention block."""
    def __init__(self, reduction=4, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        channels = input_shape[-1]
        self.squeeze = Dense(channels // self.reduction, activation="swish", name=f"{self.name}_squeeze")
        self.excite = Dense(channels, activation="sigmoid", name=f"{self.name}_excite")
        super().build(input_shape)

    def call(self, inputs):
        se = self.squeeze(inputs)
        se = self.excite(se)
        return Multiply()([inputs, se])

    def get_config(self):
        config = super().get_config()
        config.update({"reduction": self.reduction})
        return config


# ────────────────── model builder ──────────────────────────
def build_model(num_l1=2, num_l2=5, num_l3=8):
    """
    Build a branched classifier on EfficientNetV2S with:
    - SiLU/Swish activations
    - SE attention in each branch
    - Hierarchical conditioning (L1→L2→L3)
    - Residual skip in Branch 3
    - Stochastic depth via Dropout on residuals
    """
    base = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_preprocessing=True,
    )
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)

    # Deeper shared trunk with SiLU
    x = Dense(512, activation="swish")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    shared = Dense(256, activation="swish")(x)
    shared = BatchNormalization()(shared)
    shared = Dropout(0.3)(shared)

    # Branch 1: Organic vs NonOrganic
    b1 = Dense(128, activation="swish")(shared)
    b1 = BatchNormalization()(b1)
    b1 = Dropout(0.2)(b1)
    b1 = SEBlock(reduction=4, name="se_branch1")(b1)
    out1 = Dense(num_l1, activation="softmax", name="organic_output", dtype="float32")(b1)

    # Hierarchical: L1 → L2
    b2_input = Concatenate()([shared, out1])
    b2 = Dense(128, activation="swish")(b2_input)
    b2 = BatchNormalization()(b2)
    b2 = Dropout(0.3)(b2)
    b2 = SEBlock(reduction=4, name="se_branch2")(b2)
    out2 = Dense(num_l2, activation="softmax", name="material_output", dtype="float32")(b2)

    # Hierarchical: L2 → L3 with residual skip
    b3_input = Concatenate()([shared, out2])
    b3 = Dense(256, activation="swish")(b3_input)
    b3 = BatchNormalization()(b3)
    b3 = Dropout(0.3)(b3)
    b3_res = Dense(192, activation="swish")(b3)
    b3_res = BatchNormalization()(b3_res)
    b3_res = Dropout(0.2)(b3_res)
    b3_proj = Dense(128, activation="swish")(b3)
    b3_res_proj = Dense(128, activation="swish")(b3_res)
    b3 = Add()([b3_proj, b3_res_proj])
    b3 = BatchNormalization()(b3)
    b3 = SEBlock(reduction=4, name="se_branch3")(b3)
    out3 = Dense(num_l3, activation="softmax", name="subtype_output", dtype="float32")(b3)

    model = Model(inputs=base.input, outputs=[out1, out2, out3])
    return model, base


def compile_model(model, lr, use_xla=True):
    """Compile with AdamW, label-smoothed CE, and optional XLA JIT."""
    model.compile(
        optimizer=AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY),
        loss={
            "organic_output": CategoricalCrossentropy(label_smoothing=0.1),
            "material_output": CategoricalCrossentropy(label_smoothing=0.1),
            "subtype_output": CategoricalCrossentropy(label_smoothing=0.15),
        },
        loss_weights={
            "organic_output": 1.0,
            "material_output": 2.0,
            "subtype_output": 3.0,
        },
        metrics={
            "organic_output": "accuracy",
            "material_output": "accuracy",
            "subtype_output": "accuracy",
        },
        jit_compile=use_xla,
    )


# ────────── warmup + cosine decay schedule ─────────────────
def warmup_cosine_schedule(epoch, lr, initial_lr, total_epochs, warmup_epochs=3, min_lr=1e-7):
    if epoch < warmup_epochs:
        return min_lr + (initial_lr - min_lr) * (epoch / warmup_epochs)
    progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
    return min_lr + 0.5 * (initial_lr - min_lr) * (1 + np.cos(np.pi * progress))


def make_callbacks(phase: int):
    cbs = [
        EarlyStopping(
            monitor="val_loss",
            patience=8 if phase >= 2 else 5,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            patience=4 if phase >= 2 else 3,
            factor=0.2,
            min_lr=1e-7,
            verbose=1,
        ),
        ModelCheckpoint(
            "best_waste_model_v3.keras",
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
    ]
    if phase == 2:
        cbs.append(
            LearningRateScheduler(
                lambda epoch, lr: warmup_cosine_schedule(
                    epoch, lr, initial_lr=PHASE2_LR,
                    total_epochs=PHASE2_EPOCHS, warmup_epochs=WARMUP_EPOCHS,
                ),
                verbose=1,
            )
        )
    elif phase == 3:
        cbs.append(
            LearningRateScheduler(
                lambda epoch, lr: warmup_cosine_schedule(
                    epoch, lr, initial_lr=PHASE3_LR,
                    total_epochs=PHASE3_EPOCHS, warmup_epochs=2,
                ),
                verbose=1,
            )
        )
    return cbs


# ──────────── class-weight computation ─────────────────────
def compute_weights(labels, num_classes):
    unique = np.unique(labels)
    weights = compute_class_weight("balanced", classes=unique, y=labels)
    weight_dict = {int(c): float(w) for c, w in zip(unique, weights)}
    for i in range(num_classes):
        weight_dict.setdefault(i, 1.0)
    return weight_dict


# [MEM-OPT] Helper to force garbage collection between phases
def clear_memory():
    """Force garbage collection and clear TF caches."""
    gc.collect()
    tf.keras.backend.clear_session()
    gc.collect()
    print("🧹 Memory cleared")


# ───────────── evaluation & visualization ──────────────────
def plot_history(history, phase_label=""):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Training History — {phase_label}", fontsize=16)

    branches = ["organic_output", "material_output", "subtype_output"]
    titles = ["Level-1 (Organic)", "Level-2 (Material)", "Level-3 (Sub-type)"]

    for i, (branch, title) in enumerate(zip(branches, titles)):
        acc_key = f"{branch}_accuracy"
        val_acc = f"val_{branch}_accuracy"
        if acc_key in history.history:
            axes[0, i].plot(history.history[acc_key], label="train")
            axes[0, i].plot(history.history[val_acc], label="val")
            axes[0, i].set_title(f"{title} — Accuracy")
            axes[0, i].legend()
            axes[0, i].grid(True, alpha=0.3)

        loss_key = f"{branch}_loss"
        val_loss = f"val_{branch}_loss"
        if loss_key in history.history:
            axes[1, i].plot(history.history[loss_key], label="train")
            axes[1, i].plot(history.history[val_loss], label="val")
            axes[1, i].set_title(f"{title} — Loss")
            axes[1, i].legend()
            axes[1, i].grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig("training_history_v3.png", dpi=150)
    plt.show()


def evaluate_model(model, paths_test, y1_test, y2_test, y3_test):
    """Print classification reports and plot confusion matrices."""
    # [MEM-OPT] Build a small eval dataset from paths (no in-RAM images)
    eval_ds = build_eval_dataset(paths_test, y1_test, y2_test, y3_test, BATCH_SIZE)

    p1_list, p2_list, p3_list = [], [], []
    for batch_x, batch_y in eval_ds:
        preds = model.predict(batch_x, verbose=0)
        p1_list.append(preds[0])
        p2_list.append(preds[1])
        p3_list.append(preds[2])

    p1 = np.concatenate(p1_list, axis=0)
    p2 = np.concatenate(p2_list, axis=0)
    p3 = np.concatenate(p3_list, axis=0)

    pred1, pred2, pred3 = np.argmax(p1, 1), np.argmax(p2, 1), np.argmax(p3, 1)
    true1, true2, true3 = y1_test, y2_test, y3_test

    # Trim to match (last batch may be smaller)
    n = len(pred1)
    true1, true2, true3 = true1[:n], true2[:n], true3[:n]

    print("\n" + "=" * 60)
    print("LEVEL-1  Organic vs NonOrganic")
    print("=" * 60)
    print(classification_report(true1, pred1, target_names=list(LEVEL1_LABELS.keys())))

    print("=" * 60)
    print("LEVEL-2  Material Type")
    print("=" * 60)
    print(classification_report(true2, pred2, target_names=list(LEVEL2_LABELS.keys())))

    print("=" * 60)
    print("LEVEL-3  Sub-type")
    print("=" * 60)
    print(classification_report(true3, pred3, target_names=list(LEVEL3_LABELS.keys())))

    # Confusion matrices
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    for ax, true, pred, names, title in [
        (axes[0], true1, pred1, list(LEVEL1_LABELS.keys()), "Level-1"),
        (axes[1], true2, pred2, list(LEVEL2_LABELS.keys()), "Level-2"),
        (axes[2], true3, pred3, list(LEVEL3_LABELS.keys()), "Level-3"),
    ]:
        cm = confusion_matrix(true, pred)
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=names, yticklabels=names, ax=ax,
        )
        ax.set_title(f"{title} Confusion Matrix")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrices_v3.png", dpi=150)
    plt.show()

    return pred1, pred2, pred3


# ─────────────── single-image inference ────────────────────
def predict_image(model, img_path: str, show: bool = True):
    img = cv2.imread(img_path)
    if img is None:
        print(f"[ERROR] Cannot read {img_path}")
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
    img_batch = np.expand_dims(img_resized, axis=0)

    p1, p2, p3 = model.predict(img_batch, verbose=0)
    l1 = LEVEL1_NAMES[np.argmax(p1)]
    l2 = LEVEL2_NAMES[np.argmax(p2)]
    l3 = LEVEL3_NAMES[np.argmax(p3)]

    print(f"\n{'─' * 45}")
    print(f"  Image  : {img_path}")
    print(f"  Level 1: {l1:<14}  (conf {np.max(p1):.2%})")
    print(f"  Level 2: {l2:<14}  (conf {np.max(p2):.2%})")
    print(f"  Level 3: {l3:<14}  (conf {np.max(p3):.2%})")
    print(f"{'─' * 45}")

    if show:
        plt.figure(figsize=(4, 4))
        plt.imshow(img_rgb)
        plt.title(f"{l1} → {l2} → {l3}")
        plt.axis("off")
        plt.show()


# ════════════════════════ MAIN ═════════════════════════════
if __name__ == "__main__":

    total_start = time.time()

    # ── 1. Scan dataset (paths only — NO images in RAM) ───
    paths, raw_y1, raw_y2, raw_y3 = scan_dataset(DATASET_PATH)

    # Stratified split using paths (tiny memory footprint)
    (paths_train, paths_test, y1_train, y1_test,
     y2_train, y2_test, y3_train, y3_test) = train_test_split(
        paths, raw_y1, raw_y2, raw_y3,
        test_size=0.2, random_state=SEED, stratify=raw_y1,
    )
    (paths_train, paths_val, y1_train, y1_val,
     y2_train, y2_val, y3_train, y3_val) = train_test_split(
        paths_train, y1_train, y2_train, y3_train,
        test_size=0.15, random_state=SEED,
    )

    print(f"Train: {len(paths_train)}  |  Val: {len(paths_val)}  |  Test: {len(paths_test)}")

    # ── 2. Class weights ──────────────────────────────────
    cw1 = compute_weights(y1_train, 2)
    cw2 = compute_weights(y2_train, 5)
    cw3 = compute_weights(y3_train, 8)
    print(f"Class weights L1: {cw1}")
    print(f"Class weights L2: {cw2}")
    print(f"Class weights L3: {cw3}")

    # ── 3. Build tf.data pipelines ────────────────────────
    val_ds = build_eval_dataset(paths_val, y1_val, y2_val, y3_val, BATCH_SIZE)

    # ── 4. Build model ────────────────────────────────────
    model, base_model = build_model()
    model.summary()

    # ── 5. Phase 1: Frozen backbone ───────────────────────
    print("\n" + "=" * 60)
    print("PHASE 1 — Training heads (EfficientNetV2S frozen)")
    print("=" * 60)

    compile_model(model, lr=PHASE1_LR)

    train_ds_p1 = build_train_dataset(
        paths_train, y1_train, y2_train, y3_train,
        BATCH_SIZE, use_mixup=False,
    )

    p1_start = time.time()
    history1 = model.fit(
        train_ds_p1,
        epochs=PHASE1_EPOCHS,
        validation_data=val_ds,
        callbacks=make_callbacks(phase=1),
    )
    p1_time = time.time() - p1_start
    print(f"⏱ Phase 1 completed in {p1_time:.1f}s ({p1_time/60:.1f} min)")
    plot_history(history1, phase_label="Phase 1 (Frozen EfficientNetV2S)")

    # [MEM-OPT] Clean up phase 1 dataset
    del train_ds_p1
    clear_memory()

    # ── 6. Phase 2: Fine-tune last 50 layers ─────────────
    print("\n" + "=" * 60)
    print(f"PHASE 2 — Fine-tuning last {abs(FINE_TUNE_AT_P2)} layers + MixUp/CutMix")
    print("=" * 60)

    base_model.trainable = True
    for layer in base_model.layers[:FINE_TUNE_AT_P2]:
        layer.trainable = False

    trainable = sum(1 for l in model.layers if l.trainable)
    frozen = sum(1 for l in model.layers if not l.trainable)
    print(f"  Trainable layers: {trainable}  |  Frozen: {frozen}")

    compile_model(model, lr=PHASE2_LR)

    # Rebuild val_ds after clear_session
    val_ds = build_eval_dataset(paths_val, y1_val, y2_val, y3_val, BATCH_SIZE)

    train_ds_p2 = build_train_dataset(
        paths_train, y1_train, y2_train, y3_train,
        BATCH_SIZE, use_mixup=True,
    )

    p2_start = time.time()
    history2 = model.fit(
        train_ds_p2,
        epochs=PHASE2_EPOCHS,
        validation_data=val_ds,
        callbacks=make_callbacks(phase=2),
    )
    p2_time = time.time() - p2_start
    print(f"⏱ Phase 2 completed in {p2_time:.1f}s ({p2_time/60:.1f} min)")
    plot_history(history2, phase_label="Phase 2 (Fine-Tune Last 50 + MixUp/CutMix)")

    # [MEM-OPT] Clean up phase 2 dataset
    del train_ds_p2
    clear_memory()

    # ── 7. Phase 3: Fine-tune last 100 layers ────────────
    print("\n" + "=" * 60)
    print(f"PHASE 3 — Deep fine-tuning last {abs(FINE_TUNE_AT_P3)} layers")
    print("=" * 60)

    for layer in base_model.layers[:FINE_TUNE_AT_P3]:
        layer.trainable = False
    for layer in base_model.layers[FINE_TUNE_AT_P3:]:
        layer.trainable = True

    trainable = sum(1 for l in model.layers if l.trainable)
    frozen = sum(1 for l in model.layers if not l.trainable)
    print(f"  Trainable layers: {trainable}  |  Frozen: {frozen}")

    compile_model(model, lr=PHASE3_LR)

    # Rebuild val_ds after clear_session
    val_ds = build_eval_dataset(paths_val, y1_val, y2_val, y3_val, BATCH_SIZE)

    train_ds_p3 = build_train_dataset(
        paths_train, y1_train, y2_train, y3_train,
        BATCH_SIZE, use_mixup=True,
    )

    p3_start = time.time()
    history3 = model.fit(
        train_ds_p3,
        epochs=PHASE3_EPOCHS,
        validation_data=val_ds,
        callbacks=make_callbacks(phase=3),
    )
    p3_time = time.time() - p3_start
    print(f"⏱ Phase 3 completed in {p3_time:.1f}s ({p3_time/60:.1f} min)")
    plot_history(history3, phase_label="Phase 3 (Deep Fine-Tune Last 100)")

    del train_ds_p3
    clear_memory()

    # ── 8. Final evaluation ───────────────────────────────
    print("\n" + "=" * 60)
    print("FINAL EVALUATION ON TEST SET")
    print("=" * 60)

    test_ds = build_eval_dataset(paths_test, y1_test, y2_test, y3_test, BATCH_SIZE)

    results = model.evaluate(test_ds, return_dict=True)
    print(f"\nOverall test loss: {results['loss']:.4f}")
    print(
        f"  L1 acc: {results['organic_output_accuracy']:.2%}  |  "
        f"L2 acc: {results['material_output_accuracy']:.2%}  |  "
        f"L3 acc: {results['subtype_output_accuracy']:.2%}"
    )

    evaluate_model(model, paths_test, y1_test, y2_test, y3_test)

    # ── 9. Timing summary ────────────────────────────────
    total_time = time.time() - total_start
    print(f"\n{'═' * 60}")
    print(f"⏱ TRAINING TIME SUMMARY")
    print(f"{'═' * 60}")
    print(f"  Phase 1 (frozen):        {p1_time:7.1f}s  ({p1_time/60:.1f} min)")
    print(f"  Phase 2 (fine-tune 50):  {p2_time:7.1f}s  ({p2_time/60:.1f} min)")
    print(f"  Phase 3 (fine-tune 100): {p3_time:7.1f}s  ({p3_time/60:.1f} min)")
    print(f"  Total:                   {total_time:7.1f}s  ({total_time/60:.1f} min)")
    print(f"{'═' * 60}")

    print("\n✓ Done. Best model saved to best_waste_model_v3.keras")


In [ ]:
pip install fastapi==0.110.0 uvicorn==0.27.1 python-multipart==0.0.9 tensorflow==2.17.1 numpy==1.26.4 opencv-python==4.9.0.80 pyngrok==7.1.6 scikit-learn matplotlib seaborn tqdm

In [ ]:
pip install nest_asyncio

In [ ]:
"""
Waste Classification API - With ngrok Support
==============================================
Endpoint:
  - POST /predict : Classify a single image
  - GET  /ngrok-url : Get public ngrok URL

Run: python waste_api_ngrok.py
"""

import numpy as np
import cv2
from datetime import datetime
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer, Dense, Multiply
import threading
import time

# Try to import ngrok
try:
    from pyngrok import ngrok
    NGROK_AVAILABLE = True
except ImportError:
    NGROK_AVAILABLE = False
    print("⚠ pyngrok not installed. Install with: pip install pyngrok")

# ==================== Configuration ====================
MODEL_PATH = "best_waste_model_v3.keras"
IMG_SIZE = 260
NGROK_PORT = 8000


NGROK_AUTH_TOKEN = "39T00KwQ8lHREdapnSvjIod9NnW_511SGNPA874NPdrsqKpmB"  

# ==================== Custom Layers ====================
@tf.keras.utils.register_keras_serializable()
class SEBlock(Layer):
    """Squeeze-and-Excitation attention block."""
    def __init__(self, reduction=4, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        channels = input_shape[-1]
        self.squeeze = Dense(channels // self.reduction, activation="swish", 
                             name=f"{self.name}_squeeze")
        self.excite = Dense(channels, activation="sigmoid", 
                            name=f"{self.name}_excite")
        super().build(input_shape)

    def call(self, inputs):
        se = self.squeeze(inputs)
        se = self.excite(se)
        return Multiply()([inputs, se])

    def get_config(self):
        config = super().get_config()
        config.update({"reduction": self.reduction})
        return config

# ==================== Label Maps ====================
LEVEL1_NAMES = {0: "Organic", 1: "NonOrganic"}
LEVEL2_NAMES = {0: "Food", 1: "Plastic", 2: "Metal", 3: "Paper", 4: "Glass"}
LEVEL3_NAMES = {
    0: "PET", 1: "HDPE", 2: "Aluminum", 3: "Steel",
    4: "Newspaper", 5: "Cardboard", 6: "ClearGlass", 7: "BrownGlass",
}

# ==================== FastAPI App ====================
app = FastAPI(
    title="Waste Classification API",
    description="Multi-level waste classification with EfficientNetV2S",
    version="3.0.0"
)

# CORS middleware - Allow frontend to access
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global variables
_model = None
_ngrok_tunnel = None
_ngrok_url = None

# ==================== Helper Functions ====================
def load_model_global():
    """Load the trained model"""
    global _model
    if _model is None:
        try:
            _model = load_model(MODEL_PATH, custom_objects={'SEBlock': SEBlock})
            print("✓ Model loaded successfully")
        except Exception as e:
            print(f"✗ Failed to load model: {e}")
            raise
    return _model

def start_ngrok():
    """Start ngrok tunnel for public access"""
    global _ngrok_tunnel, _ngrok_url
    
    if not NGROK_AVAILABLE:
        print("⚠ pyngrok not available. Install with: pip install pyngrok")
        return None
    
    if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN":
        print("\n" + "="*60)
        print("⚠ NGROK NOT CONFIGURED!")
        print("  To expose your API publicly, please:")
        print("  1. Sign up at https://dashboard.ngrok.com/signup")
        print("  2. Get your auth token from https://dashboard.ngrok.com/auth")
        print("  3. Replace 'YOUR_NGROK_AUTH_TOKEN' in the script")
        print("="*60 + "\n")
        print("  The API will still work locally at http://localhost:8000")
        return None
    
    try:
        # Kill any existing tunnels
        ngrok.kill()
        
        # Set auth token
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        
        # Create new tunnel
        ngrok.kill()
        _ngrok_tunnel = ngrok.connect(NGROK_PORT, bind_tls=True)
        _ngrok_url = _ngrok_tunnel.public_url.replace("http://", "https://")
        
        print(f"\n{'='*60}")
        print(f"✓ ngrok tunnel established!")
        print(f"  Public URL: {_ngrok_url}")
        print(f"  API Endpoint: {_ngrok_url}/predict")
        print(f"  Health Check: {_ngrok_url}/health")
        print(f"{'='*60}\n")
        
        return _ngrok_url
    except Exception as e:
        print(f"✗ Failed to start ngrok: {e}")
        return None

def preprocess_image(image_bytes: bytes) -> np.ndarray:
    """Preprocess image for model input"""
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("Invalid image format")
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
    return img_resized

# ==================== API Endpoints ====================

@app.on_event("startup")
async def startup_event():
    """Load model and start ngrok on startup"""
    load_model_global()
    
    # Start ngrok in background
    if NGROK_AVAILABLE:
        thread = threading.Thread(target=start_ngrok, daemon=True)
        thread.start()
        # Wait a moment for ngrok to start
        time.sleep(2)
    
    print("\n" + "="*50)
    print("🚀 Waste Classification API Ready!")
    print("📍 Local URL: http://localhost:8000")
    print("📖 API Docs: http://localhost:8000/docs")
    if _ngrok_url:
        print(f"🌍 Public URL: {_ngrok_url}")
    print("="*50 + "\n")

@app.get("/health")
async def health_check():
    """Health check endpoint"""
    return {
        "status": "healthy",
        "model_loaded": _model is not None,
        "ngrok_url": _ngrok_url,
        "timestamp": datetime.now().isoformat()
    }

@app.get("/ngrok-url")
async def get_ngrok_url():
    """Get the current ngrok public URL"""
    return {
        "ngrok_url": _ngrok_url,
        "is_active": _ngrok_url is not None,
        "message": "Use this URL to access the API from anywhere" if _ngrok_url else "Ngrok not configured or not started"
    }

@app.post("/predict")
async def predict_image(file: UploadFile = File(...)):
    """
    Classify a single image
    
    Returns:
        {
            "success": true,
            "predictions": {
                "level1": {"class": "NonOrganic", "confidence": 99.45},
                "level2": {"class": "Plastic", "confidence": 94.52},
                "level3": {"class": "PET", "confidence": 92.31}
            },
            "all_probabilities": {
                "level1": {"Organic": 0.55, "NonOrganic": 99.45},
                "level2": {"Food": 0.12, "Plastic": 94.52, ...},
                "level3": {"PET": 92.31, "HDPE": 4.56, ...}
            },
            "timestamp": "2024-01-01T12:00:00"
        }
    """
    try:
        # Validate file type
        if not file.content_type.startswith("image/"):
            raise HTTPException(status_code=400, detail="File must be an image")
        
        # Validate file size (max 10MB)
        file.file.seek(0, 2)
        file_size = file.file.tell()
        file.file.seek(0)
        if file_size > 10 * 1024 * 1024:
            raise HTTPException(status_code=400, detail="File size must be less than 10MB")
        
        # Read and preprocess image
        image_bytes = await file.read()
        img_array = preprocess_image(image_bytes)
        
        # Make prediction
        model = load_model_global()
        img_batch = np.expand_dims(img_array, axis=0)
        predictions = model.predict(img_batch, verbose=0)
        
        # Parse predictions
        l1_probs = predictions[0][0]
        l2_probs = predictions[1][0]
        l3_probs = predictions[2][0]
        
        l1_class = np.argmax(l1_probs)
        l2_class = np.argmax(l2_probs)
        l3_class = np.argmax(l3_probs)
        
        # Convert to percentages
        l1_confidence = float(l1_probs[l1_class]) * 100
        l2_confidence = float(l2_probs[l2_class]) * 100
        l3_confidence = float(l3_probs[l3_class]) * 100
        
        # Build all probabilities (as percentages)
        l1_all = {LEVEL1_NAMES[i]: round(float(l1_probs[i]) * 100, 2) 
                  for i in range(len(LEVEL1_NAMES))}
        l2_all = {LEVEL2_NAMES[i]: round(float(l2_probs[i]) * 100, 2) 
                  for i in range(len(LEVEL2_NAMES))}
        l3_all = {LEVEL3_NAMES[i]: round(float(l3_probs[i]) * 100, 2) 
                  for i in range(len(LEVEL3_NAMES))}
        
        # Sort probabilities by confidence (highest first)
        l2_sorted = dict(sorted(l2_all.items(), key=lambda x: x[1], reverse=True))
        l3_sorted = dict(sorted(l3_all.items(), key=lambda x: x[1], reverse=True))
        
        # Build response
        response = {
            "success": True,
            "predictions": {
                "level1": {
                    "class": LEVEL1_NAMES[l1_class],
                    "confidence": round(l1_confidence, 2)
                },
                "level2": {
                    "class": LEVEL2_NAMES[l2_class],
                    "confidence": round(l2_confidence, 2)
                },
                "level3": {
                    "class": LEVEL3_NAMES[l3_class],
                    "confidence": round(l3_confidence, 2)
                }
            },
            "all_probabilities": {
                "level1": l1_all,
                "level2": l2_sorted,
                "level3": l3_sorted
            },
            "timestamp": datetime.now().isoformat()
        }
        
        return JSONResponse(content=response)
        
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")

# ==================== Run Server ====================
if __name__ == "__main__":
    import uvicorn
    
    print("\n" + "="*60)
    print("♻️  Waste Classification API with ngrok Support")
    print("="*60)
    
    # Check if model exists
    import os
    if not os.path.exists(MODEL_PATH):
        print(f"\n⚠ WARNING: Model file '{MODEL_PATH}' not found!")
        print("  Please make sure the model file is in the current directory.")
        print("  The API will still start but predictions will fail.\n")
    
    # Run the server
    import nest_asyncio
    import uvicorn

    nest_asyncio.apply()

    config = uvicorn.Config(app, host="0.0.0.0", port=NGROK_PORT)
    server = uvicorn.Server(config)

    await server.serve()

In [ ]:
"""
Waste Classification v3 — Explainability (SHAP + LIME + Grad-CAM)
==================================================================
Generates interpretability plots for the branched EfficientNetV2S model.

Outputs (saved to ./explainability_outputs/):
  1. Grad-CAM heatmaps for each branch (organic, material, subtype)
  2. LIME superpixel explanations per classification level
  3. SHAP GradientExplainer attribution maps
  4. SHAP summary bar plots (mean |SHAP| per class)
  5. Combined comparison grid (Original → Grad-CAM → LIME → SHAP)

Requirements:
  pip install shap lime matplotlib seaborn opencv-python tensorflow scikit-image

Usage:
  1. Make sure best_waste_model_v3.keras exists (or change MODEL_PATH)
  2. Set DATASET_PATH or IMAGE_PATHS below
  3. Run:  python Waste_classification.py
"""

# ───────────────────────── imports ─────────────────────────
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Layer, Dense, Multiply
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Explainability libraries
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

# ───────────────────── configuration ───────────────────────
MODEL_PATH = "/kaggle/working/best_waste_model_v3.keras"
DATASET_PATH = "/kaggle/input/datasets/idly1234/wastecnn/Data"  # ← change this

# Provide specific image paths for explanation, or leave empty to auto-pick
IMAGE_PATHS = []  # e.g., ["path/to/image1.jpg", "path/to/image2.jpg"]
NUM_SAMPLES = 5   # number of images to explain if IMAGE_PATHS is empty

IMG_SIZE = 260    # Fixed: Reduced from 300 to 260 to match the lowmem model
OUTPUT_DIR = "explainability_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Label maps (must match waste_classification_v3.py)
LEVEL1_LABELS = {"Organic": 0, "NonOrganic": 1}
LEVEL2_LABELS = {"Food": 0, "Plastic": 1, "Metal": 2, "Paper": 3, "Glass": 4}
LEVEL3_LABELS = {
    "PET": 0, "HDPE": 1, "Aluminum": 2, "Steel": 3,
    "Newspaper": 4, "Cardboard": 5, "ClearGlass": 6, "BrownGlass": 7,
}

LEVEL1_NAMES = {v: k for k, v in LEVEL1_LABELS.items()}
LEVEL2_NAMES = {v: k for k, v in LEVEL2_LABELS.items()}
LEVEL3_NAMES = {v: k for k, v in LEVEL3_LABELS.items()}

BRANCH_CONFIG = [
    ("organic_output",  "Level-1: Organic/NonOrganic", LEVEL1_NAMES),
    ("material_output", "Level-2: Material Type",      LEVEL2_NAMES),
    ("subtype_output",  "Level-3: Sub-type",           LEVEL3_NAMES),
]

# ──────────────────── custom layers ────────────────────────
@tf.keras.utils.register_keras_serializable()
class SEBlock(Layer):
    """Squeeze-and-Excitation attention block. Needed to load the saved model."""
    def __init__(self, reduction=4, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        channels = input_shape[-1]
        self.squeeze = Dense(channels // self.reduction, activation="swish", name=f"{self.name}_squeeze")
        self.excite = Dense(channels, activation="sigmoid", name=f"{self.name}_excite")
        super().build(input_shape)

    def call(self, inputs):
        se = self.squeeze(inputs)
        se = self.excite(se)
        return Multiply()([inputs, se])

    def get_config(self):
        config = super().get_config()
        config.update({"reduction": self.reduction})
        return config


# ───────────────────── helper functions ────────────────────
def load_and_preprocess(img_path):
    """Load an image and preprocess for model input."""
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read: {img_path}")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
    return img_rgb, img_resized


def collect_sample_images(dataset_path, num_samples=5):
    """Walk dataset and pick diverse sample images (one per category if possible)."""
    category_images = {}
    for root, _dirs, files in os.walk(dataset_path):
        for fname in files:
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                parts = root.replace("\\", "/").split("/")
                cat = parts[-1] if parts else "unknown"
                if cat not in category_images:
                    category_images[cat] = os.path.join(root, fname)

    # Pick from different categories first
    selected = list(category_images.values())[:num_samples]

    # If not enough categories, pad from the first available
    if len(selected) < num_samples:
        all_imgs = []
        for root, _dirs, files in os.walk(dataset_path):
            for fname in files:
                if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    all_imgs.append(os.path.join(root, fname))
                    if len(all_imgs) >= num_samples:
                        break
            if len(all_imgs) >= num_samples:
                break
        selected = all_imgs[:num_samples]

    return selected


# ════════════════════════════════════════════════════════════
#  1. GRAD-CAM
# ════════════════════════════════════════════════════════════
def find_last_conv_layer(model):
    """Find the last convolutional layer in the flat model."""
    for layer in reversed(model.layers):
        if isinstance(layer, (tf.keras.layers.Conv2D,)):
            return layer.name
    return None


def get_gradcam_heatmap(model, img_array, output_name, last_conv_layer_name):
    """
    Compute Grad-CAM heatmap for a specific output branch.

    Builds a sub-model: input → last_conv_layer → target_output
    Then computes gradients of the predicted class w.r.t. conv feature maps.
    """
    # Build grad model (Fixed for non-nested flat model architectures)
    try:
        last_conv_layer = model.get_layer(last_conv_layer_name).output
        grad_model = Model(
            inputs=model.input,
            outputs=[last_conv_layer, model.get_layer(output_name).output]
        )
    except Exception as e:
        print(f"[WARN] Could not find conv layer '{last_conv_layer_name}': {e}")
        return np.zeros((IMG_SIZE, IMG_SIZE))

    img_tensor = tf.cast(tf.expand_dims(img_array, 0), tf.float32)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        predicted_class = tf.argmax(predictions[0])
        class_output = predictions[:, predicted_class]

    grads = tape.gradient(class_output, conv_outputs)

    if grads is None:
        print(f"[WARN] Gradients are None for {output_name}")
        return np.zeros((IMG_SIZE, IMG_SIZE))

    # Global average pooling of gradients
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight the feature maps
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    # Convert to float32 numpy (mixed_float16 returns float16 which cv2 can't handle)
    heatmap = tf.cast(heatmap, tf.float32).numpy()

    # Ensure 2D — squeeze may collapse to scalar if conv has 1 filter
    if heatmap.ndim < 2:
        return np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)

    return heatmap.astype(np.float32)


def generate_gradcam_plots(model, img_paths, last_conv_name):
    """Generate and save Grad-CAM heatmaps for all branches on sample images."""
    print("\n🔥 Generating Grad-CAM heatmaps...")

    for img_idx, img_path in enumerate(img_paths):
        try:
            img_rgb, img_preprocessed = load_and_preprocess(img_path)
        except FileNotFoundError as e:
            print(f"  [SKIP] {e}")
            continue

        # Get predictions
        preds = model.predict(np.expand_dims(img_preprocessed, 0), verbose=0)
        pred_labels = [LEVEL1_NAMES[np.argmax(preds[0])],
                       LEVEL2_NAMES[np.argmax(preds[1])],
                       LEVEL3_NAMES[np.argmax(preds[2])]]
        pred_confs = [np.max(preds[0]), np.max(preds[1]), np.max(preds[2])]

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"Grad-CAM — {os.path.basename(img_path)}", fontsize=14, fontweight='bold')

        # Original image
        display_img = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        axes[0].imshow(display_img)
        axes[0].set_title("Original Image", fontsize=11)
        axes[0].axis("off")

        for branch_idx, (output_name, branch_title, _) in enumerate(BRANCH_CONFIG):
            heatmap = get_gradcam_heatmap(model, img_preprocessed, output_name, last_conv_name)

            # Resize heatmap to image size (ensure float32 for cv2)
            heatmap_resized = cv2.resize(heatmap.astype(np.float32), (IMG_SIZE, IMG_SIZE))
            heatmap_color = cv2.applyColorMap(
                np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
            )
            heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

            # Overlay
            overlay = (heatmap_color * 0.4 + display_img * 0.6).astype(np.uint8)

            axes[branch_idx + 1].imshow(overlay)
            axes[branch_idx + 1].set_title(
                f"{branch_title}\n{pred_labels[branch_idx]} ({pred_confs[branch_idx]:.1%})",
                fontsize=10
            )
            axes[branch_idx + 1].axis("off")

        plt.tight_layout(rect=[0, 0, 1, 0.93])
        save_path = os.path.join(OUTPUT_DIR, f"gradcam_sample_{img_idx + 1}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: {save_path}")


# ════════════════════════════════════════════════════════════
#  2. LIME EXPLANATIONS
# ════════════════════════════════════════════════════════════
def generate_lime_plots(model, img_paths):
    """Generate LIME superpixel explanations for each branch."""
    print("\n🟢 Generating LIME explanations...")

    explainer = lime_image.LimeImageExplainer(random_state=42)

    for img_idx, img_path in enumerate(img_paths):
        try:
            img_rgb, img_preprocessed = load_and_preprocess(img_path)
        except FileNotFoundError as e:
            print(f"  [SKIP] {e}")
            continue

        display_img = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        img_norm = img_preprocessed  # keep 0-255 for EfficientNetV2S

        # Get predictions
        preds = model.predict(np.expand_dims(img_preprocessed, 0), verbose=0)
        pred_labels = [LEVEL1_NAMES[np.argmax(preds[0])],
                       LEVEL2_NAMES[np.argmax(preds[1])],
                       LEVEL3_NAMES[np.argmax(preds[2])]]
        pred_confs = [np.max(preds[0]), np.max(preds[1]), np.max(preds[2])]

        fig, axes = plt.subplots(2, 4, figsize=(22, 10))
        fig.suptitle(f"LIME Explanations — {os.path.basename(img_path)}", fontsize=14, fontweight='bold')

        # Original
        axes[0, 0].imshow(display_img)
        axes[0, 0].set_title("Original Image", fontsize=11)
        axes[0, 0].axis("off")
        axes[1, 0].axis("off")

        for branch_idx, (output_name, branch_title, name_map) in enumerate(BRANCH_CONFIG):
            # Build a prediction function for this specific branch
            branch_model = Model(inputs=model.input, outputs=model.get_layer(output_name).output)

            def predict_fn(images, _bm=branch_model):
                images = images.astype(np.float32)
                return _bm.predict(images, verbose=0)

            # Run LIME
            explanation = explainer.explain_instance(
                img_norm.astype(np.uint8) if img_norm.max() > 1 else (img_norm * 255).astype(np.uint8),
                predict_fn,
                top_labels=len(name_map),
                hide_color=0,
                num_samples=500,  # reduce for speed; increase for accuracy
                random_seed=42,
            )

            predicted_class = np.argmax(preds[branch_idx])

            # Positive-only mask (regions that support the prediction)
            temp, mask = explanation.get_image_and_mask(
                predicted_class,
                positive_only=True,
                num_features=10,
                hide_rest=False,
            )
            bounded_img = mark_boundaries(temp / 255.0, mask, color=(0, 1, 0), mode='thick')

            axes[0, branch_idx + 1].imshow(bounded_img)
            axes[0, branch_idx + 1].set_title(
                f"{branch_title}\nPositive regions → {pred_labels[branch_idx]} ({pred_confs[branch_idx]:.1%})",
                fontsize=9
            )
            axes[0, branch_idx + 1].axis("off")

            # Positive + Negative mask (pro vs. con regions)
            temp2, mask2 = explanation.get_image_and_mask(
                predicted_class,
                positive_only=False,
                num_features=10,
                hide_rest=False,
            )
            bounded_img2 = mark_boundaries(temp2 / 255.0, mask2, color=(1, 0, 0), mode='thick')

            axes[1, branch_idx + 1].imshow(bounded_img2)
            axes[1, branch_idx + 1].set_title(
                f"Pro (green) vs Con (red) regions",
                fontsize=9
            )
            axes[1, branch_idx + 1].axis("off")

        plt.tight_layout(rect=[0, 0, 1, 0.93])
        save_path = os.path.join(OUTPUT_DIR, f"lime_sample_{img_idx + 1}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: {save_path}")


# ════════════════════════════════════════════════════════════
#  3. SHAP EXPLANATIONS
# ════════════════════════════════════════════════════════════
def generate_shap_plots(model, img_paths, background_images):
    """
    Generate SHAP attribution maps using GradientExplainer.

    GradientExplainer is the best option for large CNN models — it approximates
    SHAP values using expected gradients (a generalization of Integrated Gradients).
    """
    print("\n🔵 Generating SHAP explanations...")

    # Prepare background data (small subset for efficiency)
    bg_data = background_images[:10].astype(np.float32)  # Limited to 10 for performance

    for branch_idx, (output_name, branch_title, name_map) in enumerate(BRANCH_CONFIG):
        print(f"\n  Processing {branch_title}...")

        # Build single-output model for this branch
        branch_model = Model(inputs=model.input, outputs=model.get_layer(output_name).output)

        try:
            # Create SHAP explainer
            explainer = shap.GradientExplainer(branch_model, bg_data)
        except Exception as e:
            print(f"  [WARN] SHAP GradientExplainer failed for {output_name}: {e}")
            print(f"  Trying DeepExplainer as fallback...")
            try:
                explainer = shap.DeepExplainer(branch_model, bg_data)
            except Exception as e2:
                print(f"  [ERROR] Both SHAP explainers failed: {e2}")
                continue

        for img_idx, img_path in enumerate(img_paths):
            try:
                img_rgb, img_preprocessed = load_and_preprocess(img_path)
            except FileNotFoundError as e:
                print(f"    [SKIP] {e}")
                continue

            display_img = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
            test_image = np.expand_dims(img_preprocessed, 0).astype(np.float32)

            # Compute SHAP values
            try:
                shap_values = explainer.shap_values(test_image)
            except Exception as e:
                print(f"    [WARN] SHAP computation failed: {e}")
                continue

            # Get prediction
            pred = branch_model.predict(test_image, verbose=0)
            predicted_class = np.argmax(pred[0])
            confidence = np.max(pred[0])

            # shap_values shape: list of arrays, one per class
            # Each array shape: (1, H, W, 3)
            if isinstance(shap_values, list):
                sv = shap_values[predicted_class][0]  # (H, W, 3)
            else:
                sv = shap_values[0]  # (H, W, 3)

            # Sum across color channels for a single attribution map
            sv_sum = np.sum(np.abs(sv), axis=-1)  # (H, W)

            fig, axes = plt.subplots(1, 3, figsize=(16, 5))
            fig.suptitle(
                f"SHAP — {branch_title} | {os.path.basename(img_path)}\n"
                f"Predicted: {name_map[predicted_class]} ({confidence:.1%})",
                fontsize=13, fontweight='bold'
            )

            # Original
            axes[0].imshow(display_img)
            axes[0].set_title("Original Image", fontsize=11)
            axes[0].axis("off")

            # SHAP heatmap
            shap_heatmap = axes[1].imshow(sv_sum, cmap='RdBu_r', alpha=0.8)
            axes[1].set_title("SHAP Attribution Map", fontsize=11)
            axes[1].axis("off")
            plt.colorbar(shap_heatmap, ax=axes[1], fraction=0.046, pad=0.04)

            # SHAP overlay on original
            if np.max(sv_sum) > 0:
                sv_norm = sv_sum / (np.max(sv_sum) + 1e-8)
                heatmap_color = cv2.applyColorMap(np.uint8(255 * sv_norm), cv2.COLORMAP_JET)
                heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
                overlay = (heatmap_color * 0.4 + display_img * 0.6).astype(np.uint8)
            else:
                overlay = display_img
            axes[2].imshow(overlay)
            axes[2].set_title("SHAP Overlay", fontsize=11)
            axes[2].axis("off")

            plt.tight_layout(rect=[0, 0, 1, 0.90])
            save_path = os.path.join(
                OUTPUT_DIR, f"shap_{output_name}_sample_{img_idx + 1}.png"
            )
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            plt.show()
            print(f"    ✓ Saved: {save_path}")


def generate_shap_summary_bar(model, background_images, test_images):
    """
    Generate SHAP summary bar plots showing mean |SHAP value| per class.

    This shows which spatial features (averaged) contribute most to each
    class prediction across a batch of test images.
    """
    print("\n📊 Generating SHAP summary bar charts...")

    bg_data = background_images[:10].astype(np.float32)
    test_data = test_images[:10].astype(np.float32)

    for branch_idx, (output_name, branch_title, name_map) in enumerate(BRANCH_CONFIG):
        print(f"  Processing {branch_title}...")

        branch_model = Model(inputs=model.input, outputs=model.get_layer(output_name).output)

        try:
            explainer = shap.GradientExplainer(branch_model, bg_data)
            shap_values = explainer.shap_values(test_data)
        except Exception as e:
            print(f"  [WARN] SHAP summary failed for {output_name}: {e}")
            continue

        # Compute mean |SHAP| per class across all test images
        num_classes = len(name_map)
        mean_shap = []
        class_names = []

        for cls_idx in range(num_classes):
            if isinstance(shap_values, list) and cls_idx < len(shap_values):
                # Mean absolute attribution across all pixels and images
                mean_val = np.mean(np.abs(shap_values[cls_idx]))
                mean_shap.append(mean_val)
                class_names.append(name_map[cls_idx])

        if not mean_shap:
            continue

        # Bar plot
        fig, ax = plt.subplots(figsize=(10, max(4, len(class_names) * 0.6)))
        colors = sns.color_palette("viridis", len(class_names))
        bars = ax.barh(class_names, mean_shap, color=colors, edgecolor='white', linewidth=0.5)
        ax.set_xlabel("Mean |SHAP value|", fontsize=12)
        ax.set_title(f"SHAP Feature Importance — {branch_title}", fontsize=14, fontweight='bold')
        ax.invert_yaxis()

        # Add value annotations
        if mean_shap:
            max_mean = max(mean_shap)
            for bar, val in zip(bars, mean_shap):
                ax.text(bar.get_width() + max_mean * 0.02, bar.get_y() + bar.get_height() / 2,
                        f'{val:.4f}', va='center', fontsize=10)

        plt.tight_layout()
        save_path = os.path.join(OUTPUT_DIR, f"shap_summary_{output_name}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: {save_path}")


# ════════════════════════════════════════════════════════════
#  4. COMBINED COMPARISON GRID
# ════════════════════════════════════════════════════════════
def generate_comparison_grid(model, img_paths, last_conv_name, background_images):
    """
    Generate a combined side-by-side comparison:
    Original | Grad-CAM | LIME | SHAP — for each image.
    """
    print("\n🎯 Generating combined comparison grids...")

    lime_explainer = lime_image.LimeImageExplainer(random_state=42)
    bg_data = background_images[:10].astype(np.float32)

    # Use Level-2 (material) branch for the combined grid
    output_name = "material_output"
    branch_model = Model(inputs=model.input, outputs=model.get_layer(output_name).output)

    def predict_fn(images):
        return branch_model.predict(images.astype(np.float32), verbose=0)

    try:
        shap_explainer = shap.GradientExplainer(branch_model, bg_data)
    except Exception:
        shap_explainer = None
        print("  [WARN] SHAP explainer unavailable for comparison grid")

    for img_idx, img_path in enumerate(img_paths):
        try:
            img_rgb, img_preprocessed = load_and_preprocess(img_path)
        except FileNotFoundError as e:
            print(f"  [SKIP] {e}")
            continue

        display_img = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        test_image = np.expand_dims(img_preprocessed, 0).astype(np.float32)

        # Prediction
        pred = branch_model.predict(test_image, verbose=0)
        predicted_class = np.argmax(pred[0])
        confidence = np.max(pred[0])
        pred_label = LEVEL2_NAMES[predicted_class]

        fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
        fig.suptitle(
            f"Explainability Comparison — {os.path.basename(img_path)}\n"
            f"Material: {pred_label} ({confidence:.1%})",
            fontsize=14, fontweight='bold'
        )

        # 1. Original
        axes[0].imshow(display_img)
        axes[0].set_title("Original", fontsize=12, fontweight='bold')
        axes[0].axis("off")

        # 2. Grad-CAM
        heatmap = get_gradcam_heatmap(model, img_preprocessed, output_name, last_conv_name)
        heatmap_resized = cv2.resize(heatmap.astype(np.float32), (IMG_SIZE, IMG_SIZE))
        if np.max(heatmap_resized) > 0:
            heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
            heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
            overlay_gc = (heatmap_color * 0.4 + display_img * 0.6).astype(np.uint8)
        else:
            overlay_gc = display_img
        axes[1].imshow(overlay_gc)
        axes[1].set_title("Grad-CAM", fontsize=12, fontweight='bold')
        axes[1].axis("off")

        # 3. LIME
        try:
            explanation = lime_explainer.explain_instance(
                img_preprocessed.astype(np.uint8),
                predict_fn,
                top_labels=len(LEVEL2_NAMES),
                hide_color=0,
                num_samples=300,
                random_seed=42,
            )
            temp, mask = explanation.get_image_and_mask(
                predicted_class, positive_only=True,
                num_features=8, hide_rest=False,
            )
            bounded = mark_boundaries(temp / 255.0, mask, color=(0, 1, 0), mode='thick')
            axes[2].imshow(bounded)
        except Exception as e:
            axes[2].imshow(display_img)
            axes[2].text(IMG_SIZE // 2, IMG_SIZE // 2, f"LIME Error:\n{e}",
                         ha='center', va='center', color='red', fontsize=8)
        axes[2].set_title("LIME", fontsize=12, fontweight='bold')
        axes[2].axis("off")

        # 4. SHAP
        if shap_explainer is not None:
            try:
                shap_values = shap_explainer.shap_values(test_image)
                if isinstance(shap_values, list):
                    sv = shap_values[predicted_class][0]
                else:
                    sv = shap_values[0]
                sv_sum = np.sum(np.abs(sv), axis=-1)
                
                if np.max(sv_sum) > 0:
                    sv_norm = sv_sum / (np.max(sv_sum) + 1e-8)
                    shap_color = cv2.applyColorMap(np.uint8(255 * sv_norm), cv2.COLORMAP_JET)
                    shap_color = cv2.cvtColor(shap_color, cv2.COLOR_BGR2RGB)
                    overlay_shap = (shap_color * 0.4 + display_img * 0.6).astype(np.uint8)
                else:
                    overlay_shap = display_img
                axes[3].imshow(overlay_shap)
            except Exception as e:
                axes[3].imshow(display_img)
                axes[3].text(IMG_SIZE // 2, IMG_SIZE // 2, f"SHAP Error:\n{e}",
                             ha='center', va='center', color='red', fontsize=8)
        else:
            axes[3].imshow(display_img)
            axes[3].text(IMG_SIZE // 2, IMG_SIZE // 2, "SHAP unavailable",
                         ha='center', va='center', color='gray', fontsize=10)
        axes[3].set_title("SHAP", fontsize=12, fontweight='bold')
        axes[3].axis("off")

        plt.tight_layout(rect=[0, 0, 1, 0.90])
        save_path = os.path.join(OUTPUT_DIR, f"comparison_grid_{img_idx + 1}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: {save_path}")


# ════════════════════════════════════════════════════════════
#  5. PER-CLASS CONFIDENCE DISTRIBUTION
# ════════════════════════════════════════════════════════════
def plot_confidence_distributions(model, test_images, test_labels_l1, test_labels_l2, test_labels_l3):
    """
    Plot prediction confidence distributions for each branch.
    Shows how confident the model is across all test images per class.
    """
    print("\n📈 Generating confidence distribution plots...")

    predictions = model.predict(test_images, batch_size=32, verbose=0)

    for branch_idx, (output_name, branch_title, name_map) in enumerate(BRANCH_CONFIG):
        preds = predictions[branch_idx]
        max_confs = np.max(preds, axis=1)
        pred_classes = np.argmax(preds, axis=1)

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        fig.suptitle(f"Confidence Analysis — {branch_title}", fontsize=14, fontweight='bold')

        # Distribution of max confidence
        axes[0].hist(max_confs, bins=50, color='#4edea3', edgecolor='white', alpha=0.8)
        axes[0].axvline(np.mean(max_confs), color='#ff6b6b', linestyle='--',
                        label=f'Mean: {np.mean(max_confs):.3f}')
        axes[0].set_xlabel("Max Prediction Confidence", fontsize=11)
        axes[0].set_ylabel("Count", fontsize=11)
        axes[0].set_title("Confidence Distribution", fontsize=12)
        axes[0].legend(fontsize=10)
        axes[0].grid(True, alpha=0.3)

        # Per-class mean confidence
        class_confs = {}
        for cls_idx in range(len(name_map)):
            mask = pred_classes == cls_idx
            if np.any(mask):
                class_confs[name_map[cls_idx]] = np.mean(max_confs[mask])

        if class_confs:
            colors = sns.color_palette("husl", len(class_confs))
            bars = axes[1].bar(class_confs.keys(), class_confs.values(),
                               color=colors, edgecolor='white', linewidth=0.5)
            axes[1].set_ylabel("Mean Confidence", fontsize=11)
            axes[1].set_title("Per-Class Mean Confidence", fontsize=12)
            axes[1].set_ylim(0, 1.05)
            axes[1].grid(True, alpha=0.3, axis='y')

            for bar, val in zip(bars, class_confs.values()):
                axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                             f'{val:.2%}', ha='center', fontsize=9, fontweight='bold')

            plt.xticks(rotation=45, ha='right')

        plt.tight_layout(rect=[0, 0, 1, 0.93])
        save_path = os.path.join(OUTPUT_DIR, f"confidence_{output_name}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: {save_path}")


# ════════════════════════════════════════════════════════════
#  MAIN
# ════════════════════════════════════════════════════════════
if __name__ == "__main__":

    print("=" * 60)
    print("  Waste Classification v3 — Explainability Suite")
    print("  (SHAP + LIME + Grad-CAM)")
    print("=" * 60)

    # ── 1. Load model ─────────────────────────────────────
    print(f"\n📦 Loading model from {MODEL_PATH}...")
    try:
        model = load_model(MODEL_PATH, custom_objects={'SEBlock': SEBlock})
        print("  ✓ Model loaded successfully")
    except Exception as e:
        print(f"  ✗ Failed to load model: {e}")
        print("  Make sure you have trained and saved the model first.")
        exit(1)

    model.summary(print_fn=lambda x: None)  # suppress summary, just validate

    # ── 2. Find last conv layer for Grad-CAM ──────────────
    last_conv_name = find_last_conv_layer(model)
    if last_conv_name:
        print(f"  ✓ Last conv layer for Grad-CAM: {last_conv_name}")
    else:
        print("  ⚠ Could not auto-detect last conv layer, Grad-CAM may fail")
        last_conv_name = "top_conv"  # EfficientNetV2S last conv layer name

    # ── 3. Collect sample images ──────────────────────────
    if IMAGE_PATHS:
        sample_paths = IMAGE_PATHS
    else:
        print(f"\n🔍 Collecting {NUM_SAMPLES} sample images from dataset...")
        sample_paths = collect_sample_images(DATASET_PATH, NUM_SAMPLES)

    if not sample_paths:
        print("  ✗ No images found! Set IMAGE_PATHS or check DATASET_PATH.")
        exit(1)
    print(f"  ✓ Found {len(sample_paths)} images")

    # ── 4. Load background images for SHAP ────────────────
    print("\n📸 Loading background images for SHAP baseline...")
    background_imgs = []
    count = 0
    for root, _dirs, files in os.walk(DATASET_PATH):
        for fname in files:
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                try:
                    img = load_img(os.path.join(root, fname), target_size=(IMG_SIZE, IMG_SIZE))
                    background_imgs.append(img_to_array(img))
                    count += 1
                except Exception:
                    pass
            if count >= 30: # Reduced for faster execution
                break
        if count >= 30:
            break
    background_imgs = np.array(background_imgs, dtype=np.float32)
    print(f"  ✓ Loaded {len(background_imgs)} background images")

    # ── 5. Load test images for confidence analysis ───────
    test_imgs = []
    for path in sample_paths:
        try:
            _, preprocessed = load_and_preprocess(path)
            test_imgs.append(preprocessed)
        except Exception:
            pass
    test_imgs = np.array(test_imgs, dtype=np.float32) if test_imgs else background_imgs[:10]

    # ── 6. Run all explainability methods ─────────────────
    print("\n" + "=" * 60)
    print("  Running Explainability Pipeline")
    print("=" * 60)

    # 6a. Grad-CAM
    generate_gradcam_plots(model, sample_paths, last_conv_name)

    # 6b. LIME
    generate_lime_plots(model, sample_paths)

    # 6c. SHAP
    generate_shap_plots(model, sample_paths, background_imgs)

    # 6d. SHAP summary bars
    generate_shap_summary_bar(model, background_imgs, test_imgs)

    # 6e. Confidence distributions
    plot_confidence_distributions(model, test_imgs, None, None, None)

    # 6f. Combined comparison grid
    generate_comparison_grid(model, sample_paths, last_conv_name, background_imgs)

    # ── Done ──────────────────────────────────────────────
    print("\n" + "=" * 60)
    print(f"  ✅ All explainability plots saved to: {OUTPUT_DIR}/")
    print("=" * 60)
    print("\nGenerated files:")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        print(f"  📄 {f}")
